# Stage1 0bp cap-release local value labels
Frozen matched one-step training label; no fit or held account.


In [1]:
import copy,hashlib,json,os,sys,shlex,tempfile
from pathlib import Path
repo=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1];sys.path.insert(0,str(repo))
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.intraday_recovery_direct_allocation import project_legal_weights
from src.backtest.ibkr_tiered import TieredFees,settle_tiered_event
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_ssh
card=repo/'research/stage1_cap_release_0bp_local_value_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][x]==0 for x in ['new_market_requests','new_model_fits','new_native_accounts','new_bootstraps'])
state=card/'run_state.json';assert not state.exists(),'frozen label pass already started'
for p,k in [(repo/src['reference_card']/'summary.json','reference_summary_sha256'),(repo/src['physical_preflight_card']/'summary.json','physical_preflight_summary_sha256'),(repo/src['physical_preflight_card']/'result.json','physical_preflight_result_sha256'),(repo/src['tariff_plan'],'tariff_plan_sha256'),(repo/'src/workflow/intraday_recovery_direct_allocation.py','projector_module_sha256'),(repo/'src/backtest/ibkr_tiered.py','fee_module_sha256')]:assert sha(p)==src[k],str(p)
source_summary=json.loads((repo/src['reference_card']/'summary.json').read_text());preflight=json.loads((repo/src['physical_preflight_card']/'result.json').read_text());tariff=json.loads((repo/src['tariff_plan']).read_text())['tariff']
assert source_summary['recorder_id']==src['reference_recorder_id'] and source_summary['experiment_id']==src['reference_experiment_id'] and source_summary['nas_readback']['verified']
assert preflight['funded_cash_floor_executable_rows']==541
provider=(repo/'data/qlib/full_path_tiered_allocation_20261013').resolve();tracking=(repo/src['reference_card']/'qlib_recorder').resolve();os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(tracking),'default_exp_name':Path(src['reference_card']).name}})
reference=R.get_recorder(experiment_name=Path(src['reference_card']).name,recorder_id=src['reference_recorder_id'])
frame=reference.load_object('inputs/source_frame').reset_index(drop=True);bundle=reference.load_object('paths/reference_0');native=reference.load_object('native/reference_0');action=reference.load_object('inputs/action_plan')
path=bundle['replay'].reset_index(drop=True);targets=bundle['targets'].reset_index(drop=True)
assert native['audit']['status']=='passed' and native['audit']['slippage_bps']==0
assert len(frame)==len(path)==len(targets)==1061 and frame.trade_date.equals(path.trade_date) and frame.trade_date.equals(targets.trade_date)
cal=list(pd.to_datetime(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day')).date);assert cal==list(pd.to_datetime(frame.trade_date).dt.date)
spec=action['recovery_spec'];assert spec['minimum_cash_weight']==0.15 and spec['maximum_soxl_weight']==0.30
state.write_text(json.dumps({'status':'started','plan_sha256':sha(card/'plan.json'),'label_passes_started':1},indent=2)+'\n')
fees=TieredFees(tariff,slippage_bps=0);q=np.zeros(2);cash=100000.;labels=[];max_target_error=0.;max_ledger_error=0.;max_initial_state_error=0.
for i in range(len(frame)):
 row=frame.iloc[i];saved=path.iloc[i];target=targets.iloc[i][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float)
 raw=row[['soxl_weight','soxx_weight']].to_numpy(float);cap=float(saved.current_beta_cap);gap=float(3*raw[0]+raw[1]-cap)
 projected=project_legal_weights(*raw,cap,minimum_cash_weight=.15,maximum_soxl_weight=.30);err=float(np.max(np.abs(projected-target)));max_target_error=max(max_target_error,err);assert err<1e-9,(i,err)
 eligible=bool(gap>=.10 and saved.execute_open_target);mature=eligible and i+1<len(frame)
 today=row[['soxl_open','soxx_open']].to_numpy(float);raw_today=row[['raw_soxl_open','raw_soxx_open']].to_numpy(float);stamp=pd.Timestamp(row.trade_date)+pd.Timedelta(hours=9,minutes=47);nav=float(q@today+cash)
 label=np.nan;available=pd.NaT;candidate_target=None
 if mature:
  next_row=frame.iloc[i+1];reset=targets.iloc[i+1][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float)
  candidate_target=project_legal_weights(*raw,cap+.10,minimum_cash_weight=.15,maximum_soxl_weight=.30)
  branch_a_fee=copy.deepcopy(fees);branch_b_fee=copy.deepcopy(fees)
  assert branch_a_fee.monthly_shares==branch_b_fee.monthly_shares==fees.monthly_shares
  a=settle_tiered_event(q,cash,desired=target,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=branch_a_fee)
  b=settle_tiered_event(q,cash,desired=candidate_target,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=branch_b_fee)
  assert np.isclose(a['nav_before'],b['nav_before'],rtol=0,atol=1e-8) and np.isclose(a['nav_before'],nav,rtol=0,atol=1e-8)
  if b['cash']<-.0000001:raise ValueError('candidate unfunded on '+str(row.trade_date))
  tomorrow=next_row[['soxl_open','soxx_open']].to_numpy(float);raw_tomorrow=next_row[['raw_soxl_open','raw_soxx_open']].to_numpy(float);nextstamp=pd.Timestamp(next_row.trade_date)+pd.Timedelta(hours=9,minutes=47)
  ar=settle_tiered_event(a['quantities'],a['cash'],desired=reset,adjusted_prices=tomorrow,raw_prices=raw_tomorrow,stamp=nextstamp,fees=branch_a_fee)
  br=settle_tiered_event(b['quantities'],b['cash'],desired=reset,adjusted_prices=tomorrow,raw_prices=raw_tomorrow,stamp=nextstamp,fees=branch_b_fee)
  av=float(ar['quantities']@tomorrow+ar['cash']);bv=float(br['quantities']@tomorrow+br['cash']);assert np.isfinite([av,bv]).all() and min(av,bv)>0
  label=(bv-av)/nav;available=pd.Timestamp(next_row.trade_date)+pd.Timedelta(hours=9,minutes=49)
 control=settle_tiered_event(q,cash,desired=target if bool(saved.execute_open_target) else None,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=fees)
 if mature:
  err=float(np.max(np.abs(np.r_[a['quantities'],a['cash']]-np.r_[control['quantities'],control['cash']])));max_initial_state_error=max(max_initial_state_error,err);assert err<=1e-8
 err=float(np.max(np.abs(np.r_[control['quantities'],control['cash']]-saved[['ending_soxl_quantity','ending_soxx_quantity','ending_cash']].to_numpy(float))));max_ledger_error=max(max_ledger_error,err);assert err<=1e-6,(i,err)
 labels.append({'signal_date':row.signal_date,'trade_date':row.trade_date,'decision_at':row.decision_at,'eligible':eligible,'label_available_at':available,'local_fee_net_advantage':label,'raw_beta_minus_cap':gap})
 q,cash=control['quantities'],control['cash']
labels=pd.DataFrame(labels);assert len(labels)==1061 and labels.eligible.sum()==541 and labels.local_fee_net_advantage.notna().sum() in (540,541)
valid=labels[labels.local_fee_net_advantage.notna()].copy();assert (pd.to_datetime(valid.label_available_at)>pd.to_datetime(valid.decision_at)).all()
annual=[]
for year,g in labels.groupby(pd.to_datetime(labels.trade_date).dt.year):
 z=g.local_fee_net_advantage.dropna();annual.append({'year':int(year),'sessions':len(g),'eligible':int(g.eligible.sum()),'mature':len(z),'positive':int((z>0).sum()),'negative':int((z<0).sum()),'zero':int((z==0).sum()),'mean':float(z.mean()) if len(z) else None})
result={'decision':'training_only_local_fee_net_label_no_receiver_or_account','sessions':len(labels),'eligible':int(labels.eligible.sum()),'mature':len(valid),'mean_local_advantage':float(valid.local_fee_net_advantage.mean()),'median_local_advantage':float(valid.local_fee_net_advantage.median()),'positive':int((valid.local_fee_net_advantage>0).sum()),'negative':int((valid.local_fee_net_advantage<0).sum()),'zero':int((valid.local_fee_net_advantage==0).sum()),'annual':annual,'max_projected_target_parity_error':max_target_error,'max_original_cash_share_ledger_error_usd':max_ledger_error,'max_control_branch_identity_error_usd':max_initial_state_error,'label_last_unknown':str(labels.trade_date.iloc[-1].date()),'new_model_fits':0,'new_native_accounts':0,'new_market_requests':0,'label_passes':1,'reused_held_outcomes':False,'interpretation':'Local same-start and next-session common-reset 0bp Tiered labels on 2018-2022 OOF training scaffold only; not a complete policy return or a 2023+ SOXX validation.'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan);rec.save_objects(artifact_path='analysis',labels=labels,annual=annual,result=result);rec.log_metrics(mature_rows=len(valid),eligible_rows=int(labels.eligible.sum()),mean_local_advantage=result['mean_local_advantage'],new_model_fits=0,new_native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'source_recorder_id':src['reference_recorder_id'],'new_model_fits':0,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['analysis/labels','analysis/annual','analysis/result']}}
binding=json.loads((repo/src['reference_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2022-12-30');(card/'nas_binding.json').write_text(json.dumps(binding,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=repo,card=card,recorder_id=rid,experiment_id=eid)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
with tempfile.TemporaryDirectory(prefix='soxl-cap-label-readback-') as temp:
 restored=Path(temp)/'card';restored.mkdir();materialize_nas_evidence(card/'nas_binding.json',card=restored)
 rel=Path('qlib_recorder')/eid/rid/'artifacts/analysis/labels';assert sha(card/rel)==sha(restored/rel)
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':remote,'scoped_restored_labels_sha256':sha(card/rel)}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
state.write_text(json.dumps({'status':'completed','plan_sha256':sha(card/'plan.json'),'label_passes_completed':1,'recorder_id':rid},indent=2)+'\n')
print(json.dumps({'result':result,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False))

{"result": {"decision": "training_only_local_fee_net_label_no_receiver_or_account", "sessions": 1061, "eligible": 541, "mature": 540, "mean_local_advantage": 8.588219763452809e-05, "median_local_advantage": 0.00017894613396477214, "positive": 285, "negative": 255, "zero": 0, "annual": [{"year": 2018, "sessions": 53, "eligible": 18, "mature": 18, "positive": 10, "negative": 8, "zero": 0, "mean": 8.157395341140011e-05}, {"year": 2019, "sessions": 252, "eligible": 39, "mature": 39, "positive": 18, "negative": 21, "zero": 0, "mean": 4.255810478873768e-06}, {"year": 2020, "sessions": 253, "eligible": 182, "mature": 182, "positive": 110, "negative": 72, "zero": 0, "mean": 0.0002453605136320887}, {"year": 2021, "sessions": 252, "eligible": 56, "mature": 56, "positive": 33, "negative": 23, "zero": 0, "mean": 0.0005414224561447958}, {"year": 2022, "sessions": 251, "eligible": 246, "mature": 245, "positive": 114, "negative": 131, "zero": 0, "mean": -0.000123400783969734}], "max_projected_target_

# Stage1 风险上限释放：0bp 同状态局部价值训练标签

冻结的 2018–2022 原架构 OOF 训练参考有 1061 个 Qlib 交易日。541 日在决策前满足原始 beta 高于自身上限至少 0.10 且原控制执行；末日没有下一交易日共同复位，故 **540** 个标签成熟。两条路径从相同数量、现金与 Tiered 月度费率计数开始，分别执行原投影目标与上限放松 0.10 后的投影目标，下一交易日 09:47 各自扣费后复位到同一原目标。标签到下一日 09:49 才可用。原投影目标、原连续账本及控制分支的最大核对误差均为 0。

540 个局部费用后差值均值 **0.00008588**（约 +0.0086% 初始净值），正值 285、负值 255。逐年均值：2018 +0.0000816（18日）、2019 +0.0000043（39日）、2020 +0.0002454（182日）、2021 +0.0005414（56日）、**2022 −0.0001234（245日）**。这否定了“训练期每逢上限受限就统一放松”有稳定同向局部价值的直觉；没有据此选择年份、符号或阈值。

原生 Qlib Recorder `ab150237a3c14f8c83e9772248323e66` 保存 1061 行含未知掩码的标签与逐年摘要；Yanxuan NAS 归档 SHA-256 `0b4b5d867ac58fcc40bb0c916ad174089a22c8007e39d7147adcfe1e8a7f2a0b`，远端哈希和限定恢复的标签对象哈希均已核对。本卡 **0 新拟合、0 新候选完整账户、0 新行情请求**。费率是冻结的 IBKR Pro Tiered 历史投影，不是真实经纪账单；共同复位标签不传播候选风险状态，因此不能当作策略财富或 SOXX 超额。下一步须另行冻结跨年接收器、匹配对照和信息门；只有信息支持后才考虑完整 Stage1 父方案的自有资金 0bp 账户。
